In [ ]:
import subprocess, sys, os, shutil
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER','0')
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=False)
UV=shutil.which('uv') or 'uv'
def run(cmd,phase):
    p=subprocess.run(cmd,capture_output=True,text=True)
    print(phase,'-> exit',p.returncode)
    if p.returncode!=0:
        print(p.stdout[-1200:]); print(p.stderr[-1200:]); raise RuntimeError(phase+' failed')
R=['torch>=2.8.0','triton>=3.4.0','transformers==4.56.2','peft==0.20.0','trl==0.22.2',
   'datasets==5.0.1','accelerate==1.15.0','bitsandbytes==0.50.2','openai-harmony==0.0.8']
run([UV,'pip','install','--system','--python',sys.executable,'--no-cache-dir',*R],'resolver')
run([UV,'pip','install','--system','--python',sys.executable,'--no-cache-dir','--no-deps','--upgrade',
     'unsloth==2026.9.4','unsloth_zoo==2026.9.3'],'frozen-no-deps')
print('install complete')


In [ ]:
# GHARIBO EXP-002 — V1 QUALIFICATION INFERENCE (20 FROZEN QUESTIONS, INFERENCE ONLY)
#
# Same inference path as the GREEN benchmark. Gold is ABSENT from this payload;
# scoring happens locally.
#
# ROOT CAUSE OF THE >900s/ITEM INCIDENT (fixed here):
#   1. NO stopping criterion. `generate()` was called with max_new_tokens=3072 and
#      no Harmony terminator handling, so it ran to the FULL 3072-token ceiling
#      every time instead of stopping at <|return|>. That alone is ~20-40x the
#      ~600-1100 tokens a gold answer actually needs.
#   2. TWO-GPU SHARDING. Unsloth split weights 5.68 GiB per T4 with `lm_head` on
#      cuda:1, so every generated token crossed the PCIe bus for the LM head.
#      The model fits on ONE T4, so sharding only cost time.
#   3. torch.no_grad() instead of torch.inference_mode().
#
# These are RUNTIME fixes only: adapter, base model, prompts, template and
# generation semantics (greedy) are unchanged.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # FIX 2: single GPU, no cross-device lm_head
import json, time, torch, pathlib, re

PROMPTS = json.loads(r'''[{"item_id": "8babd32b23f68f892a11a994354bb0b9f31faf8311baed14f958d57f100ef48b", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:branded_by:b3a3d33af19ba0fb) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:branded_by:b3a3d33af19ba0fb\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"relationType\": \"BRANDED_BY\",\n      \"sourceExternalKey\": \"model:ubiquiti:unifi-access-current-readers-and-intercoms:g3-intercom\",\n      \"targetExternalKey\": \"brand:unifi\",\n      \"confidence\": \"HIGH\",\n      \"notes\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:ubi-access-readers\",\n        \"sourceUrl\": \"https://www.ui.com/us/en/door-access/readers\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source supports BRANDED_BY relation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "53de9c24696f4dc6696448b797e82d6cf8f6250735a0e809bcec925dbcde1043", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SERVICE, externalKey=service:security:requirements-assessment) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SERVICE\",\n    \"externalKey\": \"service:security:requirements-assessment\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Security Requirements Assessment\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SERVICE\",\n      \"serviceClass\": \"CONSULTING\",\n      \"purpose\": \"Establish functional, operational and security requirements before system design.\",\n      \"typicalActivities\": [\n        \"Review business/security goals\",\n        \"Identify required functions and constraints\",\n        \"Document design inputs\"\n      ],\n      \"typicalDeliverables\": [\n        \"Requirements basis / assessment notes\"\n      ],\n      \"applicableSystemGroups\": [\n        \"ALL_SECURITY_SYSTEMS\"\n      ],\n      \"deliveryModes\": [\n        \"On-site\",\n        \"Remote\"\n      ],\n      \"recurring\": false,\n      \"prerequisites\": [],\n      \"vendorNeutralRegistryService\": true,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"service_class\",\n        \"name\": \"Service class\",\n        \"value\": \"CONSULTING\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:services:assessment-design\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"recurring\",\n        \"name\": \"Recurring service\",\n        \"value\": false,\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"boolean\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:9851c0cef71e510286dccef3\",\n        \"sourceUrl\": \"https://www.axis.com/en-us/solutions/professional-services\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the commercial/professional service activity represented by Security Requirements Assessment.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "24e0a29c81ce87e41e89a3e92f56f16807faea3dd04b85d456dd9698ec6fd22c", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:bispectral-thermal-visible-surveillance) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:bispectral-thermal-visible-surveillance\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"Bispectral Thermal + Visible Surveillance System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"MULTISENSOR_ARCHITECTURE\",\n      \"purpose\": \"Integrated thermal-and-visible architecture combining heat-based detection with visual verification.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": \"Thermal + visible video\",\n      \"managementModel\": null,\n      \"recordingModel\": null,\n      \"typicalComponents\": [\n        \"Bispectral camera\",\n        \"VMS/NVR/cloud endpoint\",\n        \"Network\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Management support for selected bispectral device/functions\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"MULTISENSOR_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:specialized-video\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"signal_transport\",\n        \"name\": \"Signal / transport\",\n        \"value\": \"Thermal + visible video\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:c2c37b4cd66c1ca67f4d11f3\",\n        \"sourceUrl\": \"https://www.axis.com/en-us/products/thermal-cameras\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by Bispectral Thermal + Visible Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "3ecaa5bf2a483e759828185538d8c852161e8dd48d73a545d2accfcf4899d9cf", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:tp-link:vigi-surveillance-kits) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:tp-link:vigi-surveillance-kits\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"TP-Link\",\n      \"brand\": \"VIGI\",\n      \"family\": \"VIGI Surveillance Kits\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Video Surveillance\",\n      \"lifecycle\": null,\n      \"name\": \"VIGI Surveillance Kits\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:1ab9db79aab2ef3751fe\",\n        \"sourceUrl\": \"https://www.vigi.com/us/vigi-app/product-list/\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping VIGI Surveillance Kits.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "148a15a72b3e5019d2004ae8d2161826004c9777cce5e3b191fd2291968f0ed9", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:parent_of:127cd3aece6f926f4628) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:parent_of:127cd3aece6f926f4628\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"PARENT_OF\",\n      \"sourceExternalKey\": \"family:dahua:dahua-project-exclusive-network-cameras\",\n      \"targetExternalKey\": \"model:dahua:dahua-project-exclusive-network-cameras:ipc-hdbw5859z-zhe-pv-pro-atc-v1\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"Dahua Project Exclusive Network Cameras contains the verified model IPC-HDBW5859Z-ZHE-PV-PRO-ATC(V1).\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:0dad3ea5b811829b322fcc59\",\n        \"sourceUrl\": \"https://previous.dahuasecurity.com/Products/All-Products/Dedicated-Products/Project-Exclusive/Network-Cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Dahua Project Exclusive Network Cameras contains the verified model IPC-HDBW5859Z-ZHE-PV-PRO-ATC(V1).\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "bc6b27535c32ccad2181688c5a9dbc2b73ae484332dc4165ab03b4d383eccdd5", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=CATEGORY, externalKey=category:global:agriculture-and-agribusiness) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"CATEGORY\",\n    \"externalKey\": \"category:global:agriculture-and-agribusiness\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Agriculture & Agribusiness\",\n      \"registryLayer\": \"CATEGORY\",\n      \"description\": \"Production, handling and commercial systems for crops, livestock, forestry and agricultural operations.\",\n      \"commercialBoundary\": \"Excludes downstream industrial food processing when the main activity is manufacturing rather than primary agriculture.\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"A\",\n      \"parentCategoryExternalKey\": null,\n      \"coreDomainCount\": 13,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [\n      \"Agriculture\",\n      \"Agribusiness\"\n    ],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:5d9274fae97bea2eec53af8b\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the economic/procurement market scope used to define Agriculture & Agribusiness.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:dc501fd3ca87942e3b2858a3\",\n        \"sourceUrl\": \"https://www.census.gov/naics/\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"2022 North American Industry Classification System supports the economic/procurement market scope used to define Agriculture & Agribusiness.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:c6182860157e4cfd1d146b40\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the economic/procurement market scope used to define Agriculture & Agribusiness.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "09d6e1a3b9a383959aa2739cde1e71cff21d2b317a6185e22bc8851db54136f6", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:component_of:0e8c39fa677fe6015066) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:component_of:0e8c39fa677fe6015066\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"COMPONENT_OF\",\n      \"sourceExternalKey\": \"model:i-pro:i-pro-s-series:wv-s65340-z2\",\n      \"targetExternalKey\": \"system:security:ip-video-surveillance\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"WV-S65340-Z2 is commercially relevant as a component of IP Video Surveillance System.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:1413a89618589216737eb920\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=S-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"WV-S65340-Z2 is commercially relevant as a component of IP Video Surveillance System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "344d7efe331d476e7da2457ae988c2ce510d72e928caff618d60dcb32f640a7a", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:suprema:door-module) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:suprema:door-module\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Suprema\",\n      \"brand\": \"Suprema\",\n      \"family\": \"Door Module\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Access I/O Module\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"Door Module\",\n      \"description\": \"Suprema current product line listed in the official hardware selector.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-selector2\",\n        \"sourceUrl\": \"https://supremainc.com/en/hardware/product_selector.asp?iCTG_No=&iPage=2&iPageSize=20\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the Door Module product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "c02027ccd0d5d578f5fb8550eade916563694bde9234a2c4aec4ea01de9ad7da", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:electric-motors-and-drives) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:electric-motors-and-drives\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Electric Motors & Drives\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Electric Motors & Drives is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on electric motors & drives; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Energy & Power\",\n        \"Industrial & Manufacturing\",\n        \"Oil, Gas & Petrochemicals\",\n        \"Water & Wastewater\"\n      ],\n      \"likelySystemFamilies\": [\n        \"LV motors\",\n        \"MV motors\",\n        \"Variable-frequency drives\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"B\",\n      \"standardsOrClassificationEvidence\": [\n        \"ECLASS Basic 16.0 Public Content Search\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:f42b54ebe17530c6ce9661d1\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the technical/commercial scope represented by Core Domain Electric Motors & Drives.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:d283c9710af0d99ac154188e\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Electric Motors & Drives.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "69797dbc393bc043080c4f3c10f46e213c45b190547197b0756ea44f78547d5e", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:site-remediation) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:site-remediation\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Site Remediation\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Site Remediation is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on site remediation; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Construction\",\n        \"Environmental & Waste Management\",\n        \"Government & Public Sector\",\n        \"Industrial & Manufacturing\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Contaminated-site cleanup\",\n        \"Groundwater remediation\",\n        \"Soil remediation\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"C\",\n      \"standardsOrClassificationEvidence\": [\n        \"UN International Standard Industrial Classification (ISIC Rev. 5)\",\n        \"2022 North American Industry Classification System\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:425248a022b6302b5b3b3e94\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the technical/commercial scope represented by Core Domain Site Remediation.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:545a6363a1ffac46a2689f8e\",\n        \"sourceUrl\": \"https://www.census.gov/naics/\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"2022 North American Industry Classification System supports the technical/commercial scope represented by Core Domain Site Remediation.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "8eef568bf2f88d75e0639022e57d86c70d085bc6ce27eedd0de0a67cf7898a8e", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:cranes-and-lifting-equipment) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:cranes-and-lifting-equipment\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Cranes & Lifting Equipment\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Cranes & Lifting Equipment is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on cranes & lifting equipment; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Construction\",\n        \"Industrial & Manufacturing\",\n        \"Logistics & Warehousing\",\n        \"Marine & Maritime\",\n        \"Mining & Minerals\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Hoists\",\n        \"Mobile lifting\",\n        \"Overhead cranes\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"B\",\n      \"standardsOrClassificationEvidence\": [\n        \"ECLASS Basic 16.0 Public Content Search\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:513b170e8621f73d39633aa3\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the technical/commercial scope represented by Core Domain Cranes & Lifting Equipment.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:c40d43bedb9adf3b5e27d8d0\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Cranes & Lifting Equipment.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "baddf953b98db8c6817bebb6a009f7636bb42afd20e8b32e3bdfa50455be2195", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=SYSTEM, externalKey=system:security:high-assurance-encrypted-access-control) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"SYSTEM\",\n    \"externalKey\": \"system:security:high-assurance-encrypted-access-control\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n    \"payload\": {\n      \"name\": \"High-Assurance Encrypted Access Control System\",\n      \"domain\": \"Security Systems\",\n      \"registryLayer\": \"SYSTEM\",\n      \"architectureKind\": \"HIGH_ASSURANCE_ARCHITECTURE\",\n      \"purpose\": \"End-to-end protected access architecture using encrypted credentials, readers/interfaces, secure I/O/gateways and access software.\",\n      \"lifecycle\": \"CURRENT\",\n      \"signalTransport\": null,\n      \"managementModel\": null,\n      \"recordingModel\": null,\n      \"typicalComponents\": [\n        \"Access software\",\n        \"Secure gateway/controller\",\n        \"Secure I/O\",\n        \"Transparent/secure reader\",\n        \"Smart credential\",\n        \"Secure Access Module where required\"\n      ],\n      \"requiredComponentsOrConditions\": [\n        \"Validated component chain and cryptographic configuration\"\n      ],\n      \"optionalComponents\": [],\n      \"protocols\": [\n        \"TLS\",\n        \"OSDP Secure Channel where applicable\"\n      ],\n      \"dependencies\": [],\n      \"topology\": null,\n      \"capacityConsiderations\": [],\n      \"designConstraints\": [],\n      \"installationConsiderations\": [],\n      \"licensingServiceDependencies\": [],\n      \"interoperabilityConstraints\": [],\n      \"standardsMaturity\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"architecture_kind\",\n        \"name\": \"Architecture kind\",\n        \"value\": \"HIGH_ASSURANCE_ARCHITECTURE\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"registry_group\",\n        \"name\": \"Registry group\",\n        \"value\": \"category:security:systems:access-control\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"lifecycle\",\n        \"name\": \"Architecture lifecycle\",\n        \"value\": \"CURRENT\",\n        \"unit\": null,\n        \"group\": \"Registry\",\n        \"dataType\": \"string\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:security-registry:3d7bf0f932fb0a413ed3af82\",\n        \"sourceUrl\": \"https://www.genetec.com/products/unified-security/synergis/high-assurance-access-control\",\n        \"sourceType\": \"OFFICIAL_OR_STANDARDS_BODY\",\n        \"observedAt\": \"2026-09-04T23:06:59+03:00\",\n        \"claim\": \"Source supports the architecture or core technical pattern represented by High-Assurance Encrypted Access Control System.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "6462d145e3203cc278b32cf86411e9633a55b38291a7b8b73d5643b0f0eaee64", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:axis:axis-p47-series) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:axis:axis-p47-series\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Axis Communications\",\n      \"brand\": \"AXIS\",\n      \"family\": \"AXIS P47 Series\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Panoramic Camera\",\n      \"lifecycle\": null,\n      \"name\": \"AXIS P47 Series\",\n      \"description\": \"Official-source family/series or normalized manufacturer catalog grouping used to organize verified model identities.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:e56e212a4f877e84d9be\",\n        \"sourceUrl\": \"https://www.axis.com/products/network-cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports family/catalog grouping AXIS P47 Series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "f79093858d95484630c986501f3f7536766fa68f636a0b1df640673372d18fc7", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:child_of:a3c6ba2365d9eb9d682b) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:child_of:a3c6ba2365d9eb9d682b\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"CHILD_OF\",\n      \"sourceExternalKey\": \"model:i-pro:i-pro-x-series:wv-x25700a-v2ln\",\n      \"targetExternalKey\": \"family:i-pro:i-pro-x-series\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"WV-X25700A-V2LN is a child model of i-PRO X Series.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:4da51ac94e4c77e0bcdde364\",\n        \"sourceUrl\": \"https://i-pro.com/products_and_solutions/en/surveillance/products-list?series=X-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"WV-X25700A-V2LN is a child model of i-PRO X Series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "c1a67234d4ac4c4442f831d803b810c80954902f6f32c47b307d912dd63500bd", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_MODEL, externalKey=model:hikvision:ds-76-recorder-family:ds-7632ni-i2) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_MODEL\",\n    \"externalKey\": \"model:hikvision:ds-76-recorder-family:ds-7632ni-i2\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Hikvision\",\n      \"brand\": \"Hikvision\",\n      \"family\": \"DS-76 Recorder Family\",\n      \"series\": \"DS-76 Recorder Family\",\n      \"model\": \"DS-7632NI-I2\",\n      \"modelNumber\": \"DS-7632NI-I2\",\n      \"mpn\": null,\n      \"sku\": null,\n      \"gtin\": null,\n      \"officialProductTitle\": \"DS-7632NI-I2\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Video Management / Recording\",\n      \"subcategory\": null,\n      \"productRole\": \"Video recorder / NVR\",\n      \"systemRole\": \"NVR-based IP Surveillance System\",\n      \"lifecycle\": \"UNKNOWN\",\n      \"name\": \"DS-7632NI-I2\",\n      \"description\": null,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [\n      {\n        \"type\": \"MODEL_NUMBER\",\n        \"value\": \"DS-7632NI-I2\",\n        \"issuer\": \"Hikvision\"\n      }\n    ],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:model:53c59e37a28bb0459b4fab60\",\n        \"sourceUrl\": \"https://opensource.hikvision.com/Home/List?id=12&page=5\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"Official source supports product/model identity DS-7632NI-I2.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "f5f969041765fc44f36f85135aaf6f3048b422b516b96e669bf932e33ba6c1e2", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=RELATION, externalKey=relation:belongs_to_category:7a5fd211b817f70948df) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"RELATION\",\n    \"externalKey\": \"relation:belongs_to_category:7a5fd211b817f70948df\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n    \"payload\": {\n      \"relationType\": \"BELONGS_TO_CATEGORY\",\n      \"sourceExternalKey\": \"family:axis:axis-m55-series\",\n      \"targetExternalKey\": \"category:security:ptz-camera\",\n      \"confidence\": \"HIGH\",\n      \"notes\": \"AXIS M55 Series is classified within PTZ Camera for this campaign.\"\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:batch003:family:7445752a2722c9b21d1c\",\n        \"sourceUrl\": \"https://www.axis.com/products/ptz-cameras\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T22:36:04+03:00\",\n        \"claim\": \"AXIS M55 Series is classified within PTZ Camera for this campaign.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "a745f758cec35f90d11adb1190baa29031364162c3d538efa96e6e34cc1db255", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:suprema:suprema-discontinued-access-products) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:suprema:suprema-discontinued-access-products\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Suprema\",\n      \"brand\": \"Suprema\",\n      \"family\": \"Suprema Discontinued Access Products\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Access Control\",\n      \"lifecycle\": \"LEGACY\",\n      \"name\": \"Suprema Discontinued Access Products\",\n      \"description\": \"Official Suprema discontinued-products area lists these historical access-control products.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:suprema-eol\",\n        \"sourceUrl\": \"https://www.supremainc.com/en/hardware/eol_biostation.asp\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the Suprema Discontinued Access Products product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "fccf19dc33a210800e97729031c9cc47b38289214472e85d3bf69ad92438afff", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=CATEGORY, externalKey=category:global:oil-gas-and-petrochemicals) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"CATEGORY\",\n    \"externalKey\": \"category:global:oil-gas-and-petrochemicals\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Oil, Gas & Petrochemicals\",\n      \"registryLayer\": \"CATEGORY\",\n      \"description\": \"Upstream, midstream, refining, gas/LNG and petrochemical commercial/engineering ecosystems.\",\n      \"commercialBoundary\": \"Separated from general energy because process equipment, hazardous-area systems and hydrocarbon logistics form a distinct market.\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"A\",\n      \"parentCategoryExternalKey\": null,\n      \"coreDomainCount\": 33,\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [\n      \"Oil & Gas\",\n      \"Petroleum\",\n      \"Petrochemical\"\n    ],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:39c575ae8f93ddfa1e09385c\",\n        \"sourceUrl\": \"https://unstats.un.org/unsd/classifications/Econ/isic/4\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UN International Standard Industrial Classification (ISIC Rev. 5) supports the economic/procurement market scope used to define Oil, Gas & Petrochemicals.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:64d700dabdd332b5e0bd4fa7\",\n        \"sourceUrl\": \"https://eclass.eu/en/eclass-standard/search-content\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"ECLASS Basic 16.0 Public Content Search supports the economic/procurement market scope used to define Oil, Gas & Petrochemicals.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:9e3ec5df1f59f4a98fa290b2\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the economic/procurement market scope used to define Oil, Gas & Petrochemicals.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "2200c67d32bb98465bc48ff12a98ecc9444fd6d946693a489814f53bb2211bb8", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=DOMAIN, externalKey=domain:global:grain-handling-and-storage) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"DOMAIN\",\n    \"externalKey\": \"domain:global:grain-handling-and-storage\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n    \"payload\": {\n      \"name\": \"Grain Handling & Storage\",\n      \"registryLayer\": \"CORE_DOMAIN\",\n      \"technicalCommercialDefinition\": \"Grain Handling & Storage is a reusable commercial/technical specialization in the VOKA global library, intended to group real systems, manufacturers, products and services around this functional scope.\",\n      \"boundary\": \"Commercial/technical scope centered on grain handling & storage; adjacent domains remain separate where their primary function differs.\",\n      \"parentCategories\": [\n        \"Agriculture & Agribusiness\",\n        \"Food & Beverage Processing\",\n        \"Logistics & Warehousing\"\n      ],\n      \"likelySystemFamilies\": [\n        \"Drying\",\n        \"Grain conveyors\",\n        \"Silos\"\n      ],\n      \"relevantGlobalServiceFamilies\": [\n        \"service:global-family:commissioning\",\n        \"service:global-family:engineering-design\",\n        \"service:global-family:installation\",\n        \"service:global-family:preventive-maintenance\",\n        \"service:global-family:repair\",\n        \"service:global-family:supply\",\n        \"service:global-family:technical-consultancy\"\n      ],\n      \"lifecycleRelevance\": \"CURRENT\",\n      \"discoveryConfidence\": \"HIGH\",\n      \"registryState\": \"STABLE_FOR_V1\",\n      \"discoveredInPass\": \"G\",\n      \"standardsOrClassificationEvidence\": [\n        \"UNGM Transportation and Storage Services\",\n        \"UNSPSC on United Nations Global Marketplace\"\n      ],\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:global-map:b23a391a770bf4c87ae62123\",\n        \"sourceUrl\": \"https://www.ungm.org/Shared/KnowledgeCenter/Pages/PC_TranStorMail\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNGM Transportation and Storage Services supports the technical/commercial scope represented by Core Domain Grain Handling & Storage.\",\n        \"confidence\": \"HIGH\"\n      },\n      {\n        \"evidenceKey\": \"evidence:global-map:ccdb19f0b95674886f5c91d7\",\n        \"sourceUrl\": \"https://www.ungm.org/Public/UNSPSC\",\n        \"sourceType\": \"AUTHORITATIVE_CLASSIFICATION\",\n        \"observedAt\": \"2026-09-05T11:07:40+03:00\",\n        \"claim\": \"UNSPSC on United Nations Global Marketplace supports the technical/commercial scope represented by Core Domain Grain Handling & Storage.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}, {"item_id": "a79fd66492ad7dee8ea3d6a6b4e171869bb6e9935c012859828c0aee82197db2", "messages": [{"role": "system", "content": "You are GHARIBO, a commercial intelligence extraction system operating on the VOKA Universal Commercial Library (UCL) framework. Your task is to process commercial product/security domain source records and produce normalized, verified structured outputs.\n\nPROCESS (follow strictly):\n1. SOURCE — Identify the origin of the input data\n2. UNDERSTAND — Parse and comprehend the entity's commercial context\n3. EXTRACT — Pull key fields: name, identifiers, attributes, aliases\n4. CLASSIFY — Determine the entity type (CATEGORY, DOMAIN, SYSTEM, MANUFACTURER, BRAND, PRODUCT_FAMILY, PRODUCT_MODEL, ITEM, SERVICE, RELATION, MARKET_RELEVANCE)\n5. NORMALIZE — Standardize field names, values, and units\n6. RELATE — Identify and validate relationships to other entities\n7. GROUND — Link claims to evidence with source URLs and confidence levels\n8. VALIDATE — Check referential integrity, required fields, and provenance\n9. STRUCTURED OUTPUT — Produce the canonical JSON record"}, {"role": "user", "content": "{\n  \"task\": \"Process the following UCL source record (entityType=PRODUCT_FAMILY, externalKey=family:moxa:moxa-tn-g4500-series) and produce the canonical normalized output.\",\n  \"sourceRecord\": {\n    \"schemaVersion\": \"1.0\",\n    \"entityType\": \"PRODUCT_FAMILY\",\n    \"externalKey\": \"family:moxa:moxa-tn-g4500-series\",\n    \"sourceRecordId\": null,\n    \"sourceUpdatedAt\": null,\n    \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n    \"payload\": {\n      \"manufacturer\": \"Moxa\",\n      \"brand\": \"Moxa\",\n      \"family\": \"Moxa TN-G4500 Series\",\n      \"domain\": \"Security Systems\",\n      \"category\": \"Industrial PoE Switch\",\n      \"lifecycle\": \"CURRENT\",\n      \"name\": \"Moxa TN-G4500 Series\",\n      \"description\": \"Official Moxa industrial Ethernet series relevant to IP surveillance/security network transport.\",\n      \"reviewFlag\": null\n    },\n    \"identifiers\": [],\n    \"aliases\": [],\n    \"attributes\": [\n      {\n        \"code\": \"environment\",\n        \"name\": \"Target environment\",\n        \"value\": \"Railway / rugged transportation\",\n        \"unit\": null,\n        \"group\": \"Environmental\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"poe_standards\",\n        \"name\": \"PoE standards\",\n        \"value\": \"IEEE 802.3af/at\",\n        \"unit\": null,\n        \"group\": \"Ethernet/PoE\",\n        \"dataType\": \"string\"\n      },\n      {\n        \"code\": \"operating_temp_min\",\n        \"name\": \"Operating temperature minimum\",\n        \"value\": -40,\n        \"unit\": \"°C\",\n        \"group\": \"Environmental\",\n        \"dataType\": \"number\"\n      },\n      {\n        \"code\": \"operating_temp_max\",\n        \"name\": \"Operating temperature maximum\",\n        \"value\": 70,\n        \"unit\": \"°C\",\n        \"group\": \"Environmental\",\n        \"dataType\": \"number\"\n      }\n    ],\n    \"marketRelevance\": [],\n    \"evidence\": [\n      {\n        \"evidenceKey\": \"evidence:moxa-tng4500\",\n        \"sourceUrl\": \"https://www.moxa.com/en/products/industrial-network-infrastructure/ethernet-switches/layer-2-managed-switches/tn-g4500-series\",\n        \"sourceType\": \"MANUFACTURER_OFFICIAL\",\n        \"observedAt\": \"2026-09-04T21:17:52+03:00\",\n        \"claim\": \"Official source enumerates or identifies the Moxa TN-G4500 Series product family/series.\",\n        \"confidence\": \"HIGH\"\n      }\n    ]\n  },\n  \"processSteps\": [\n    \"1. SOURCE — identify origin\",\n    \"2. UNDERSTAND — parse commercial context\",\n    \"3. EXTRACT — pull key fields\",\n    \"4. CLASSIFY — determine entity type\",\n    \"5. NORMALIZE — standardize fields\",\n    \"6. RELATE — validate relationships\",\n    \"7. GROUND — link to evidence\",\n    \"8. VALIDATE — check integrity\",\n    \"9. OUTPUT — canonical JSON\"\n  ]\n}"}]}]''')
assert len(PROMPTS) == 20, 'expected exactly 20 frozen items'
assert all(set(p.keys()) == {'item_id','messages'} for p in PROMPTS), 'gold leaked into payload'
print('frozen qualification items:', len(PROMPTS), '| gold present: False')

WORKING = pathlib.Path('/kaggle/working')
ADAPTER = None
for c in sorted(pathlib.Path('/kaggle/input').rglob('adapter_model.safetensors')):
    if 'checkpoint' not in str(c):
        ADAPTER = c.parent; break
assert ADAPTER is not None, 'adapter not found'
print('adapter dir:', ADAPTER)

t0 = time.time()
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER), max_seq_length=3072, load_in_4bit=True, full_finetuning=False)
ADAPTER_LOAD = time.time() - t0
MODEL_LOAD = ADAPTER_LOAD

FastLanguageModel.for_inference(model)
model.eval()
print('device map:', getattr(model, 'hf_device_map', 'n/a'))
print('lm_head device:', next(model.lm_head.parameters()).device if hasattr(model,'lm_head') else 'n/a')

# FIX 1: explicit Harmony terminators + stopping criteria.
TERMINATORS = ['<|return|>', '<|call|>']
STOP_IDS = set()
for tok in TERMINATORS:
    ids = tokenizer.encode(tok, add_special_tokens=False)
    if ids: STOP_IDS.add(ids[-1])
if tokenizer.eos_token_id is not None: STOP_IDS.add(tokenizer.eos_token_id)
print('stop token ids:', sorted(STOP_IDS))
from transformers import StoppingCriteria, StoppingCriteriaList
class HarmonyStop(StoppingCriteria):
    def __init__(self, ids): self.ids = ids
    def __call__(self, input_ids, scores, **kw):
        return all(int(input_ids[i][-1]) in self.ids for i in range(input_ids.shape[0]))

def render(msgs):
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
        reasoning_effort='medium', strftime_now=lambda _f: '2026-09-15')

def final_channel(text):
    last = None
    for m in re.finditer(r'<\|channel\|>([A-Za-z_][A-Za-z0-9_]*)\s*<\|message\|>', text):
        s = m.end(); e = len(text)
        for t in ('<|return|>','<|end|>','<|call|>','<|start|>'):
            i = text.find(t, s)
            if i != -1: e = min(e, i)
        if m.group(1) == 'final': last = text[s:e]
    return last

# Warm-up (excluded from the gate; first call pays CUDA/kernel JIT costs).
_ = tokenizer(render(PROMPTS[0]['messages']), return_tensors='pt', add_special_tokens=False).to('cuda')
with torch.inference_mode():
    model.generate(**_ , max_new_tokens=16, do_sample=False,
                   pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
print('warm-up done')

rows = []
for i, item in enumerate(PROMPTS):
    t_start = time.time()
    ids = tokenizer(render(item['messages']), return_tensors='pt', add_special_tokens=False).to('cuda')
    prompt_tokens = int(ids['input_ids'].shape[1])
    torch.cuda.synchronize(); t_prefill = time.time()
    with torch.inference_mode():                       # FIX 3
        out = model.generate(**ids, max_new_tokens=3072, do_sample=False,
                             eos_token_id=tokenizer.eos_token_id,
                             pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                             stopping_criteria=StoppingCriteriaList([HarmonyStop(STOP_IDS)]))
    torch.cuda.synchronize(); t_gen = time.time()
    new_ids = out[0][prompt_tokens:]
    raw = tokenizer.decode(new_ids, skip_special_tokens=False)
    t_ext = time.time()
    ans = final_channel(raw)
    t_end = time.time()
    gen_tokens = int(new_ids.shape[0])
    rows.append({'item_id': item['item_id'], 'prompt_tokens': prompt_tokens,
        'generated_tokens': gen_tokens,
        'prefill_seconds': round(t_prefill - t_start, 3),
        'generation_seconds': round(t_gen - t_prefill, 3),
        'tokens_per_second': round(gen_tokens / max(t_gen - t_prefill, 1e-6), 3),
        'extraction_seconds': round(t_end - t_ext, 4),
        'total_seconds': round(t_end - t_start, 3),
        'termination_reason': 'STOP_TOKEN' if int(new_ids[-1]) in STOP_IDS else 'MAX_NEW_TOKENS',
        'final_channel_found': ans is not None, 'ok': ans is not None,
        'raw': raw, 'prediction': ans})
    r = rows[-1]
    print('ITEM %d/%d id=%s prompt_tok=%d gen_tok=%d prefill=%.2fs gen=%.2fs tok/s=%.2f total=%.2fs term=%s final=%s ok=%s'
          % (i+1, len(PROMPTS), r['item_id'][:12], r['prompt_tokens'], r['generated_tokens'],
             r['prefill_seconds'], r['generation_seconds'], r['tokens_per_second'],
             r['total_seconds'], r['termination_reason'], r['final_channel_found'], r['ok']))
    (WORKING/'predictions.jsonl').write_text(
        '\n'.join(json.dumps(x, ensure_ascii=False) for x in rows)+'\n', encoding='utf-8')

totals = sorted(r['total_seconds'] for r in rows)
median = totals[len(totals)//2]
tps = sum(r['tokens_per_second'] for r in rows)/len(rows)
verdict = 'GREEN' if median <= 180 else ('YELLOW' if median <= 300 else 'RED')
summary = {'MODEL_LOAD_SECONDS': round(MODEL_LOAD,2), 'ADAPTER_LOAD_SECONDS': round(ADAPTER_LOAD,2),
  'GPU_LAYOUT': str(getattr(model,'hf_device_map','n/a')),
  'PEAK_GPU_MEMORY_GiB': round(torch.cuda.max_memory_allocated()/1024**3, 3),
  'rows': rows, 'median_item_seconds': round(median,3),
  'mean_tokens_per_second': round(tps,3), 'final_channel_pass': sum(1 for r in rows if r['final_channel_found']),
  'VERDICT': verdict}
(WORKING/'qualification-summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
(WORKING/'COMPLETED').write_text('GHARIBO-exp-002-qualification' + chr(10), encoding='utf-8')
print(json.dumps({k:v for k,v in summary.items() if k!='rows'}, indent=2))
print('COMPLETE')
